# Model-Free Reinforcement Learning for Market Making on Limit Order Books
### Implementation & Empirical Backtest of IJCAI-20 (Paper 0615)
**Reference**: *Proceedings of the Twenty-Ninth International Joint Conference on Artificial Intelligence (IJCAI-20), Special Track on AI in FinTech*

---

## 1. Executive Summary & Theoretical Framework

Market-making firms provide liquidity by posting resting limit buy (bid) and limit sell (ask) orders into the **Limit Order Book (LOB)**, earning the bid-ask spread whenever counterparties execute against both sides of their quotes. However, market makers face two fundamental risks:
1. **Adverse Selection**: Incoming aggressive market orders may possess directional information, filling the quote just before prices move unfavorably.
2. **Inventory Risk**: Accumulating large directional inventory exposes the firm to adverse market price drift, requiring risk penalties and inventory liquidation.

### Core Contributions of the Paper:
* **Finite-Horizon Discrete-Time MDP**: Formulates the high-frequency market-making problem with at most one unit resting on the best bid and best ask ($A_t \in \{(0,0), (0,1), (1,0), (1,1)\}$).
* **Conservative Queue Priority (Back-of-the-Queue)**: Conservatively assumes our submitted limit order sits at the end of the price queue, filling only when incoming market order volume completely exhausts all prior resting orders.
* **State Aggregation Function ($10^{60} \to 200$ States)**: Mitigates the curse of dimensionality by projecting high-dimensional LOB levels into 5 microstructural and portfolio attributes:
  1. $\text{bidSpeed } (BS) \in \{0, 1\}$
  2. $\text{askSpeed } (AS) \in \{0, 1\}$
  3. $\text{avgmidChangeFrac } (MF) \in \{0, \pm 1, \pm 2\}$
  4. $\text{invSign } (IS) \in \{0, \pm 1, \pm 2\}$
  5. $\text{cumPnL } (CP) \in \{0, 1\}$
* **Admissible Action Space Restriction ($\mathcal{A}_s$)**: When inventory is heavily unbalanced ($IS = \pm 2$), orders that would worsen the position are strictly disabled, capping catastrophic drawdowns.
* **Q-Learning with Historical Sample Paths (Algorithm 1)**: Updates the $640$-parameter Q-table using historical sample paths matching market conditions $(\text{BS}, \text{AS}, \text{MF})$.
* **Realistic Risk Controls**: Daily stop-loss and maximum drawdown circuits, evaluated via in-sample parameter tuning and strictly out-of-sample backtesting.


In [ ]:
import math
import itertools
from dataclasses import dataclass, replace
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Styling setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Environment initialized successfully.")


## 2. Mathematical Model & Definitions

### 2.1 State Representation & Timing
At decision epoch $t \in \{0, 1, \dots, T\}$:
* Price levels $p \in \mathcal{P} := \{1, \dots, n\}$
* $R_{t,p}^1 \in \{0, 1\}$ denotes whether our order rests at price $p$.
* $R_{t,p}^2 \ge 0$ denotes resting volume from other participants.
* Best Bid: $\underline{R}_t := \max \{p : R_{t,p}^1 + R_{t,p}^2 < 0\}$ (negative by bid-side sign convention)
* Best Ask: $\bar{R}_t := \min \{p : R_{t,p}^1 + R_{t,p}^2 > 0\}$
* Mid Price: $m(R_t) := \frac{\underline{R}_t + \bar{R}_t}{2}$

### 2.2 Objective & Reward Function (Eq. 6 - 8)
Between time $t$ and $t+1$, exogenous market buy $\hat{D}_{MB,t}$ and market sell $\hat{D}_{MS,t}$ arrive:
$$\text{Spread Capture PnL: } C(R_t, A_t, \hat{D}_t) = E^\text{bid} \cdot (m(R_t) - \underline{R}_t) + E^\text{ask} \cdot (\bar{R}_t - m(R_t))$$
where $E^\text{bid} = \mathbf{1}\{A_{t,1} = 1 \text{ and } \hat{D}_{MS,t} \ge R_{t,\underline{R}_t}^1 + R_{t,\underline{R}_t}^2 + 1\}$ under the conservative back-of-queue rule.

Accounting for mid-price drift on open inventory $inv_t$:
$$V(R_t, A_t, \hat{D}_t, inv_t) = C(R_t, A_t, \hat{D}_t) + inv_{t+1} \cdot (m(R_{t+1}) - m(R_t))$$

### 2.3 State Discretization & Action Restriction
The 5 aggregated features:
* $BS = \mathbf{1}\{\hat{D}_{MS,t} > R_{t,\underline{R}_t}^2\}$
* $AS = \mathbf{1}\{\hat{D}_{MB,t} > R_{t,\bar{R}_t}^2\}$
* $MF \in \{0, \pm 1, \pm 2\}$: $f_t = \frac{\Delta \bar{m}_t}{\text{range}(m_{[t-1, t]})}$, $|MF| = 2$ if $|f_t| > f$, $1$ if $0 < |f_t| \le f$, and $0$ if $f_t = 0$.
* $IS \in \{0, \pm 1, \pm 2\}$: $|IS| = 2$ if $|inv_t| > I$, $1$ if $0 < |inv_t| \le I$, and $0$ if $inv_t = 0$.
* $CP \in \{0, 1\}$: $CP = 1$ if $pnl_t \le P$ (underperforming session), $0$ otherwise.

Restricted action space $\mathcal{A}_s$ (Eq. 11):
$$\mathcal{A}_s = \\begin{cases}
\{(0,0), (1,0)\} & \\text{if } IS = -2 \\text{ (severely short: only allow buys/cancels)} \\\\
\{(0,0), (0,1)\} & \\text{if } IS = +2 \\text{ (severely long: only allow sells/cancels)} \\\\
\{(0,0), (0,1), (1,0), (1,1)\} & \\text{otherwise}
\\end{cases}$$


In [ ]:
@dataclass
class MarketMakingConfig:
    # State Discretization Thresholds (Section 4.1)
    f: float = 0.5           # Mid-price change fraction threshold
    I: float = 5.0           # Inventory boundary for extreme exposure (IS = +/-2)
    P: float = 0.0           # Cumulative session PnL threshold (CP = 1 if pnl <= P)
    
    # Q-Learning Hyperparameters (Section 4.2 - 4.3)
    gamma: float = 0.90      # Temporal discount factor
    alpha0: float = 1.0      # Base learning rate: alpha = alpha0 / (1 + K(s,a))
    n_iters: int = 1500      # Q-learning training iterations
    subset_frac: float = 0.5 # Fraction of state-action pairs updated per iteration
    pnl_sample_range: float = 20.0
    
    # Risk Management & Circuit Breakers (Section 5.1)
    pnl_stop: float = -50.0       # Max daily loss circuit breaker (flatten & halt)
    max_drawdown: float = 60.0    # Intraday peak-to-trough drawdown halt
    spread: float = 1.0           # Fixed spread in ticks
    seed: int = 42

cfg = MarketMakingConfig()
print(f"Config initialized: f={cfg.f}, I={cfg.I}, P={cfg.P}, gamma={cfg.gamma}, alpha0={cfg.alpha0}")


## 3. Data Engine: Limit Order Book Simulator

Section 3 of the paper highlights that empirical order volumes in Level II LOB data follow heavy-tailed distributions (failing standard thin-tail Gaussian fits). 

Below, we construct an efficient 1-second aggregated LOB data container and synthetic market generator calibrated to reproduce:
* Heavy-tailed order arrival sizes (Pareto distributed)
* Stochastic queue depth at best bid and best ask
* Endogenous mid-price revisions driven by net order flow pressure


In [ ]:
@dataclass
class LOBBars:
    bid_q: np.ndarray       # Resting queue depth at best bid (excluding our order)
    ask_q: np.ndarray       # Resting queue depth at best ask (excluding our order)
    dms: np.ndarray         # Incoming market sell demand arriving in [t, t+1)
    dmb: np.ndarray         # Incoming market buy demand arriving in [t, t+1)
    mid_open: np.ndarray    # Mid price at beginning of second t
    mid_close: np.ndarray   # Mid price at end of second t
    mid_hi: np.ndarray      # Highest mid price observed in second t
    mid_lo: np.ndarray      # Lowest mid price observed in second t
    day_id: np.ndarray      # Day index for chronological walk-forward cross validation

    def __len__(self):
        return len(self.dms)

    @property
    def mid_avg(self) -> np.ndarray:
        return 0.5 * (self.mid_open + self.mid_close)

    def slice(self, start_idx: int, end_idx: int) -> 'LOBBars':
        return LOBBars(
            bid_q=self.bid_q[start_idx:end_idx],
            ask_q=self.ask_q[start_idx:end_idx],
            dms=self.dms[start_idx:end_idx],
            dmb=self.dmb[start_idx:end_idx],
            mid_open=self.mid_open[start_idx:end_idx],
            mid_close=self.mid_close[start_idx:end_idx],
            mid_hi=self.mid_hi[start_idx:end_idx],
            mid_lo=self.mid_lo[start_idx:end_idx],
            day_id=self.day_id[start_idx:end_idx]
        )

def generate_synthetic_lob(n_days: int = 25, day_len: int = 3600, seed: int = 42) -> LOBBars:
    """Generates realistic 1-second LOB bars calibrated with heavy-tailed Pareto arrivals."""
    rng = np.random.default_rng(seed)
    n = n_days * day_len

    # Pareto heavy-tailed volume distributions
    def pareto_volumes(alpha: float, scale: float, size: int) -> np.ndarray:
        return np.floor(scale * rng.pareto(alpha, size)).astype(int)

    bid_q = 1 + pareto_volumes(2.2, 4.0, n)
    ask_q = 1 + pareto_volumes(2.2, 4.0, n)
    dms = pareto_volumes(2.0, 3.0, n)
    dmb = pareto_volumes(2.0, 3.0, n)

    mid = np.empty(n + 1, dtype=float)
    mid[0] = 1000.0  # Base tick level
    hi = np.empty(n, dtype=float)
    lo = np.empty(n, dtype=float)

    for t in range(n):
        # Microstructure price movement: buy pressure lifts price; sell pressure depresses price
        delta = 0.0
        if dmb[t] > ask_q[t] and dms[t] <= bid_q[t]:
            delta = 1.0
        elif dms[t] > bid_q[t] and dmb[t] <= ask_q[t]:
            delta = -1.0
        
        mid[t + 1] = mid[t] + delta
        hi[t] = max(mid[t], mid[t + 1])
        lo[t] = min(mid[t], mid[t + 1])

    day_id = np.repeat(np.arange(n_days), day_len)

    return LOBBars(
        bid_q=bid_q,
        ask_q=ask_q,
        dms=dms,
        dmb=dmb,
        mid_open=mid[:-1].copy(),
        mid_close=mid[1:].copy(),
        mid_hi=hi,
        mid_lo=lo,
        day_id=day_id
    )

print("Simulating sample LOB data...")
sample_lob = generate_synthetic_lob(n_days=10, day_len=1800, seed=42)
print(f"Generated {len(sample_lob):,} seconds across {len(np.unique(sample_lob.day_id))} trading days.")


## 4. State Aggregation & Environment Transitions

We implement:
1. `compute_market_features`: Computes $BS, AS, MF$ strictly using past and contemporaneous data (zero look-ahead leakage).
2. `allowed_actions`: Enforces the restricted action space $\mathcal{A}_s$ from Equation (11).
3. `step_env`: Implements Equations (1)–(8) with back-of-the-queue execution mechanics.


In [ ]:
# Action tuples: (bid_action, ask_action)
# 0 = do nothing / cancel; 1 = place limit order at best price
ACTIONS = ((0, 0), (0, 1), (1, 0), (1, 1))
ACTION_LABELS = ['Cancel All (0,0)', 'Ask Only (0,1)', 'Bid Only (1,0)', 'Both Sides (1,1)']

# Q-Table dimension: (BS: 2, AS: 2, MF: 5, IS: 5, CP: 2, Action: 4)
Q_SHAPE = (2, 2, 5, 5, 2, 4)

def compute_market_features(bars: LOBBars, f: float) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Computes bidSpeed (BS), askSpeed (AS), and avgmidChangeFrac (MF)."""
    bs = (bars.dms > bars.bid_q).astype(int)
    as_ = (bars.dmb > bars.ask_q).astype(int)
    
    avg_mid = bars.mid_avg
    n = len(bars)
    mf = np.zeros(n, dtype=int)
    
    d_mid = avg_mid[1:] - avg_mid[:-1]
    mid_range = np.maximum(bars.mid_hi[1:], bars.mid_hi[:-1]) - np.minimum(bars.mid_lo[1:], bars.mid_lo[:-1])
    
    with np.errstate(divide='ignore', invalid='ignore'):
        f_t = np.where(mid_range > 0, d_mid / mid_range, 0.0)
    
    magnitude = np.where(f_t == 0, 0, np.where(np.abs(f_t) > f, 2, 1))
    mf[1:] = np.sign(f_t).astype(int) * magnitude
    
    return bs, as_, mf

def inv_to_state_idx(inv: int, I: float) -> int:
    """Maps inventory level to IS in {-2, -1, 0, +1, +2} -> array index {0, 1, 2, 3, 4}."""
    abs_inv = abs(inv)
    if abs_inv == 0:
        val = 0
    else:
        sign = int(np.sign(inv))
        val = sign * (2 if abs_inv > I else 1)
    return val + 2

def cum_pnl_to_state_idx(pnl: float, P: float) -> int:
    """Maps cumulative session PnL to CP in {0, 1}."""
    return 1 if pnl <= P else 0

def get_state_tuple(bs_t: int, as_t: int, mf_t: int, inv: int, pnl: float, cfg: MarketMakingConfig) -> Tuple[int, int, int, int, int]:
    return (
        int(bs_t),
        int(as_t),
        int(mf_t) + 2,
        inv_to_state_idx(inv, cfg.I),
        cum_pnl_to_state_idx(pnl, cfg.P)
    )

def get_allowed_actions(is_raw: int) -> List[int]:
    """Implements Equation (11) from paper:
    If IS == -2 (oversold): only (0,0) and (1,0) [Bid only]
    If IS == +2 (overbought): only (0,0) and (0,1) [Ask only]
    Otherwise: all actions {(0,0), (0,1), (1,0), (1,1)}
    """
    if is_raw == -2:
        return [0, 2]
    elif is_raw == 2:
        return [0, 1]
    return [0, 1, 2, 3]

def step_env(bars: LOBBars, t: int, action: Tuple[int, int], current_inv: int) -> Tuple[float, int, bool, bool]:
    """Simulates 1-second transition under conservative back-of-the-queue priority (Eq. 1-8)."""
    a_bid, a_ask = action
    next_t = t + 1
    
    bid_depth = bars.bid_q[next_t]
    ask_depth = bars.ask_q[next_t]
    
    # Back-of-the-queue condition: demand must exhaust all existing orders PLUS our 1 unit
    bid_fill = bool(a_bid and bars.dms[next_t] >= bid_depth + 1)
    ask_fill = bool(a_ask and bars.dmb[next_t] >= ask_depth + 1)
    
    # Half-spread capture profit C (Eq. 6)
    half_spread = 0.5
    c_pnl = (half_spread if bid_fill else 0.0) + (half_spread if ask_fill else 0.0)
    
    # Update inventory (Eq. 7)
    next_inv = current_inv + int(bid_fill) - int(ask_fill)
    
    # Inventory mid-price drift penalty (Eq. 8)
    delta_mid = bars.mid_close[next_t] - bars.mid_close[t]
    reward_v = c_pnl + (next_inv * delta_mid)
    
    return reward_v, next_inv, bid_fill, ask_fill


## 5. Q-Learning Algorithm with Historical Sampling (Algorithm 1)

Algorithm 1 overcomes the lack of an exact transition matrix by **stochastic sample path approximation**:
1. Groups historical timestamps into candidate subsets $\tau_s$ indexed by market attributes $(BS, AS, MF)$.
2. In each iteration $n$, samples a random subset of valid state-action pairs $(s, a)$.
3. Draws a historical sample path $\omega \in \tau_s$, initializes randomized inventory and PnL consistent with $(IS, CP)$.
4. Simulates one step, translates to new state $s'$, and performs the stochastic Q-update:
$$Q_{n+1}(s, a) = (1 - \alpha_n(s,a)) Q_n(s, a) + \alpha_n(s, a) \left[ V(\omega, a) + \gamma \max_{v \in \mathcal{A}_{s'}} Q_n(s', v) \right]$$
with Robbins-Monro decaying learning rate $\alpha_n(s, a) = \frac{\alpha_0}{1 + K_n(s, a)}$.


In [ ]:
class QLearningMarketMaker:
    def __init__(self, bars: LOBBars, cfg: MarketMakingConfig):
        self.bars = bars
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg.seed)
        
        self.Q = np.zeros(Q_SHAPE, dtype=float)
        self.K = np.zeros(Q_SHAPE, dtype=int)
        
        self.bs, self.as_, self.mf = compute_market_features(bars, cfg.f)
        
        # Build tau_s: map market states (BS, AS, MF) to valid historical timestamps
        self.tau: Dict[Tuple[int, int, int], np.ndarray] = {}
        for t in range(1, len(bars) - 1):
            key = (int(self.bs[t]), int(self.as_[t]), int(self.mf[t]) + 2)
            self.tau.setdefault(key, []).append(t)
        self.tau = {k: np.array(v) for k, v in self.tau.items()}
        
        # Enumerate all valid (state, action) pairs satisfying the action restriction
        self.pairs: List[Tuple[Tuple[int, int, int, int, int], int]] = []
        for bs in (0, 1):
            for as_ in (0, 1):
                for mf in range(5):
                    m_key = (bs, as_, mf)
                    if m_key not in self.tau:
                        continue
                    for is_idx in range(5):
                        is_raw = is_idx - 2
                        allowed = get_allowed_actions(is_raw)
                        for cp in (0, 1):
                            s = (bs, as_, mf, is_idx, cp)
                            for a in allowed:
                                self.pairs.append((s, a))
        
        print(f"QLearner initialized with {len(self.pairs):,} admissible state-action pairs.")

    def _sample_consistent_inventory(self, is_raw: int) -> int:
        if is_raw == 0:
            return 0
        I = int(self.cfg.I)
        if abs(is_raw) == 1:
            val = self.rng.integers(1, max(2, I + 1))
        else: # |IS| == 2
            val = self.rng.integers(I + 1, max(I + 2, 2 * I + 1))
        return int(np.sign(is_raw) * val)

    def _sample_consistent_pnl(self, cp: int) -> float:
        rng_span = self.cfg.pnl_sample_range
        if cp == 1: # pnl <= P
            return self.cfg.P - float(self.rng.uniform(0.0, rng_span))
        else: # pnl > P
            return self.cfg.P + float(self.rng.uniform(1e-4, rng_span))

    def train(self, log_interval: int = 250) -> List[float]:
        cfg = self.cfg
        m = max(1, int(cfg.subset_frac * len(self.pairs)))
        loss_history = []
        
        for n in range(cfg.n_iters):
            sampled_indices = self.rng.choice(len(self.pairs), size=m, replace=False)
            total_delta = 0.0
            
            for idx in sampled_indices:
                s, a = self.pairs[idx]
                m_key = (s[0], s[1], s[2])
                t_sample = int(self.rng.choice(self.tau[m_key]))
                
                inv_sample = self._sample_consistent_inventory(s[3] - 2)
                pnl_sample = self._sample_consistent_pnl(s[4])
                
                # Execute environment transition
                action_tuple = ACTIONS[a]
                v_reward, next_inv, _, _ = step_env(self.bars, t_sample, action_tuple, inv_sample)
                next_pnl = pnl_sample + v_reward
                
                # Form new state
                next_t = t_sample + 1
                s_next = get_state_tuple(
                    self.bs[next_t], self.as_[next_t], self.mf[next_t],
                    next_inv, next_pnl, cfg
                )
                
                # Learning rate update
                self.K[s + (a,)] += 1
                alpha = cfg.alpha0 / (1.0 + self.K[s + (a,)])
                
                # Bellman optimal target
                next_is_raw = s_next[3] - 2
                allowed_next_actions = get_allowed_actions(next_is_raw)
                max_next_q = max(self.Q[s_next + (b,)] for b in allowed_next_actions)
                
                old_q = self.Q[s + (a,)]
                new_q = (1.0 - alpha) * old_q + alpha * (v_reward + cfg.gamma * max_next_q)
                self.Q[s + (a,)] = new_q
                total_delta += abs(new_q - old_q)
            
            mean_delta = total_delta / m
            loss_history.append(mean_delta)
            if log_interval and (n % log_interval == 0 or n == cfg.n_iters - 1):
                print(f"Iteration {n:4d}/{cfg.n_iters} | Mean |dQ|: {mean_delta:.5f}")
                
        return loss_history

    def get_policy_action(self, s: Tuple[int, int, int, int, int]) -> int:
        """Selects argmax_{a in A_s} Q(s, a)."""
        is_raw = s[3] - 2
        allowed = get_allowed_actions(is_raw)
        return max(allowed, key=lambda a: self.Q[s + (a,)])


## 6. Backtesting Simulator & Risk Controls

The backtesting engine replays a designated trading policy second-by-second with:
* Real-time position tracking and spread capture
* **Daily session reset**: Positions and cumulative PnL reset at the close of each day
* **External Risk Circuit Breakers (Section 5.1)**:
  * Stop Loss: If intraday PnL drops below `pnl_stop`, position is liquidated at mid and trading halts for the day.
  * Max Drawdown Breaker: If peak-to-current intraday drawdown exceeds `max_drawdown`, position is flattened and halted.


In [ ]:
def run_backtest(bars: LOBBars, cfg: MarketMakingConfig, policy_fn) -> Dict:
    """Replays policy across all bars and calculates risk-adjusted performance."""
    bs, as_, mf = compute_market_features(bars, cfg.f)
    n = len(bars)
    
    daily_pnls = {}
    total_fills = 0
    all_inventories = []
    equity_curve = []
    
    t = 0
    running_equity = 0.0
    
    while t < n - 1:
        current_day = bars.day_id[t]
        inv = 0
        pnl = 0.0
        peak_pnl = 0.0
        circuit_broken = False
        
        while t < n - 1 and bars.day_id[t] == current_day and bars.day_id[t + 1] == current_day:
            if not circuit_broken:
                s = get_state_tuple(bs[t], as_[t], mf[t], inv, pnl, cfg)
                action_idx = policy_fn(s)
                
                # Step market
                reward_v, inv, bf, af = step_env(bars, t, ACTIONS[action_idx], inv)
                pnl += reward_v
                running_equity += reward_v
                total_fills += (int(bf) + int(af))
                peak_pnl = max(peak_pnl, pnl)
                
                # Check circuit breakers
                if pnl < cfg.pnl_stop or (peak_pnl - pnl) > cfg.max_drawdown:
                    inv = 0
                    circuit_broken = True
            
            all_inventories.append(inv)
            equity_curve.append(running_equity)
            t += 1
            
        daily_pnls[current_day] = pnl
        t += 1 # advance past day boundary
        
    daily_arr = np.array([daily_pnls[d] for d in sorted(daily_pnls.keys())])
    sharpe = 0.0
    if len(daily_arr) > 1 and daily_arr.std(ddof=1) > 0:
        sharpe = (daily_arr.mean() / daily_arr.std(ddof=1)) * math.sqrt(252)
        
    eq_arr = np.array(equity_curve)
    peak_eq = np.maximum.accumulate(eq_arr) if len(eq_arr) > 0 else np.array([0.0])
    drawdowns = peak_eq - eq_arr if len(eq_arr) > 0 else np.array([0.0])
    max_dd = float(np.max(drawdowns)) if len(drawdowns) > 0 else 0.0
    
    return {
        'total_pnl': float(daily_arr.sum()),
        'sharpe': float(sharpe),
        'max_drawdown': max_dd,
        'fills': total_fills,
        'mean_abs_inv': float(np.mean(np.abs(all_inventories))) if all_inventories else 0.0,
        'daily_pnl': daily_arr,
        'equity_curve': eq_arr,
        'inventories': np.array(all_inventories)
    }


## 7. Empirical Experiment: Tuning & Out-of-Sample Testing

To strictly prevent data leakage and look-ahead bias:
1. **Train Set (Days 0–14)**: Used exclusively for Q-learning policy optimization.
2. **Validation Set (Days 15–19)**: Used for hyperparameter tuning across threshold grids $(f, I, P)$.
3. **Held-Out Test Set (Days 20–24)**: Evaluated **exactly once** on unseen out-of-sample data.

We benchmark our Q-learning agent against three realistic baseline policies:
* **Always-Quote (Standard MM)**: Consistently quotes both bid and ask (as permitted by $\mathcal{A}_s$).
* **Random Agent**: Uniformly selects an admissible action from $\mathcal{A}_s$.
* **Inventory Skew Heuristic**: Quotes both sides when neutral ($IS = 0$); cancels the side that would worsen inventory when skewed.


In [ ]:
print("Preparing chronological dataset splits...")
full_data = generate_synthetic_lob(n_days=25, day_len=1800, seed=101)

# Chronological partition
split_train = 15 * 1800
split_val   = 20 * 1800
data_train = full_data.slice(0, split_train)
data_val   = full_data.slice(split_train, split_val)
data_test  = full_data.slice(split_val, len(full_data))

print(f"Train set: {len(data_train):,} bars (15 days)")
print(f"Val set:   {len(data_val):,} bars (5 days)")
print(f"Test set:  {len(data_test):,} bars (5 days)")

# In-sample parameter tuning
print("\n--- Starting In-Sample Parameter Grid Search ---")
best_cfg = None
best_sharpe = -float('inf')
tuning_results = []

grid_f = [0.3, 0.6]
grid_I = [3.0, 6.0]
grid_P = [-10.0, 0.0]

for f_val, I_val, P_val in itertools.product(grid_f, grid_I, grid_P):
    trial_cfg = replace(cfg, f=f_val, I=I_val, P=P_val, n_iters=400)
    agent = QLearningMarketMaker(data_train, trial_cfg)
    agent.train(log_interval=0)
    
    val_res = run_backtest(data_val, trial_cfg, agent.get_policy_action)
    tuning_results.append({
        'f': f_val, 'I': I_val, 'P': P_val,
        'val_pnl': val_res['total_pnl'],
        'val_sharpe': val_res['sharpe']
    })
    print(f"Params: f={f_val}, I={I_val}, P={P_val} | Val PnL: {val_res['total_pnl']:6.1f} | Val Sharpe: {val_res['sharpe']:6.2f}")
    if val_res['sharpe'] > best_sharpe:
        best_sharpe = val_res['sharpe']
        best_cfg = trial_cfg

tuning_df = pd.DataFrame(tuning_results)
print(f"\nOptimal Configuration Selected: f={best_cfg.f}, I={best_cfg.I}, P={best_cfg.P} (Sharpe: {best_sharpe:.2f})")


In [ ]:
# Train final model on Train set using optimal hyperparameters
final_cfg = replace(best_cfg, n_iters=1000)
print("\n--- Training Final Q-Learning Model ---")
final_agent = QLearningMarketMaker(data_train, final_cfg)
training_losses = final_agent.train(log_interval=200)

# Define baseline policies
def always_quote_policy(s):
    # Selects maximum allowed action (prefers both sides = action 3)
    return max(get_allowed_actions(s[3] - 2))

rng_baseline = np.random.default_rng(999)
def random_policy(s):
    return int(rng_baseline.choice(get_allowed_actions(s[3] - 2)))

def inventory_skew_policy(s):
    is_raw = s[3] - 2
    if is_raw < 0: # Long desired
        return 2 # Bid only
    elif is_raw > 0: # Short desired
        return 1 # Ask only
    return 3 # Both sides

# Evaluate all policies on the out-of-sample test set
policies = {
    'Q-Learning (Paper)': final_agent.get_policy_action,
    'Always-Quote': always_quote_policy,
    'Inventory-Skew Heuristic': inventory_skew_policy,
    'Random Agent': random_policy
}

oos_results = {}
for name, pol_fn in policies.items():
    res = run_backtest(data_test, final_cfg, pol_fn)
    oos_results[name] = res

# Display summary table
summary_rows = []
for name, res in oos_results.items():
    summary_rows.append({
        'Strategy': name,
        'Total PnL': f"{res['total_pnl']:.1f}",
        'Annualized Sharpe': f"{res['sharpe']:.2f}",
        'Max Drawdown': f"{res['max_drawdown']:.1f}",
        'Total Fills': res['fills'],
        'Mean |Inventory|': f"{res['mean_abs_inv']:.2f}"
    })

pd.DataFrame(summary_rows)


## 8. Visualizations & Policy Microstructure Diagnostics

The paper outlines four key economic properties of the learned optimal Q-learning policy:
1. **Quote the Active Side**: When order flow tilts sell-heavy ($BS=1, AS=0$), placing limit buys captures flow while mitigating ask side toxic fills.
2. **Non-Directionality**: Mid-price changes ($MF$) do not induce speculative directional bets.
3. **Inventory Mean Reversion**: The policy dynamically adapts quoting to pull inventory towards zero.
4. **Drawdown Protection**: Under degraded session PnL ($CP=1$), quoting is restrained to preserve capital.

Below, we plot the convergence trajectory, equity curves, inventory distributions, and Q-table policy slices.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Training Convergence
axes[0, 0].plot(training_losses, color='#1f77b4', lw=1.5)
axes[0, 0].set_title('Q-Learning Convergence: Mean |dQ| per Iteration', fontweight='bold')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('Mean |dQ|')
axes[0, 0].set_yscale('log')
axes[0, 0].grid(True, which='both', ls='--', alpha=0.5)

# 2. Out-of-Sample Equity Curves
colors = {'Q-Learning (Paper)': '#2ca02c', 'Always-Quote': '#d62728', 
          'Inventory-Skew Heuristic': '#ff7f0e', 'Random Agent': '#7f7f7f'}
for name, res in oos_results.items():
    axes[0, 1].plot(res['equity_curve'], label=name, color=colors[name], lw=2.0 if 'Q-Learning' in name else 1.2)
axes[0, 1].set_title('Out-of-Sample Cumulative Equity Curves (Test Set)', fontweight='bold')
axes[0, 1].set_xlabel('Elapsed Time (Seconds)')
axes[0, 1].set_ylabel('Cumulative PnL (Ticks)')
axes[0, 1].legend(loc='lower left')
axes[0, 1].grid(True, ls='--', alpha=0.5)

# 3. Inventory Distribution Comparison
axes[1, 0].hist(oos_results['Q-Learning (Paper)']['inventories'], bins=np.arange(-10, 11) - 0.5, 
                density=True, alpha=0.6, color='#2ca02c', label='Q-Learning', edgecolor='black')
axes[1, 0].hist(oos_results['Always-Quote']['inventories'], bins=np.arange(-10, 11) - 0.5, 
                density=True, alpha=0.4, color='#d62728', label='Always-Quote', edgecolor='black')
axes[1, 0].set_title('Out-of-Sample Inventory Distribution Comparison', fontweight='bold')
axes[1, 0].set_xlabel('Inventory Units (inv_t)')
axes[1, 0].set_ylabel('Empirical Probability Density')
axes[1, 0].legend()
axes[1, 0].grid(True, ls='--', alpha=0.5)

# 4. Q-Table Action Value Slice (Inventory vs Action)
# Slice: BS=0, AS=0, MF=0 (neutral mid), CP=0 (normal PnL)
is_labels = ['Oversold (-2)', 'Mild Short (-1)', 'Neutral (0)', 'Mild Long (+1)', 'Overbought (+2)']
q_slice = final_agent.Q[0, 0, 2, :, 0, :] # Shape: (5 IS levels, 4 Actions)

for a_idx, label in enumerate(ACTION_LABELS):
    axes[1, 1].plot(range(5), q_slice[:, a_idx], marker='o', label=label, lw=1.8)
axes[1, 1].set_xticks(range(5))
axes[1, 1].set_xticklabels(is_labels)
axes[1, 1].set_title('Learned Q-Values vs Inventory Level (BS=0, AS=0, MF=0, CP=0)', fontweight='bold')
axes[1, 1].set_xlabel('Inventory State (IS)')
axes[1, 1].set_ylabel('Estimated Q-Factor')
axes[1, 1].legend()
axes[1, 1].grid(True, ls='--', alpha=0.5)

plt.tight_layout()
plt.show()


## 9. Next Steps: Connecting to Real Level II Market Data

This notebook provides the complete algorithmic foundation conforming to IJCAI-20 Paper 0615. To deploy on real financial instruments (e.g. CME equity futures or Vietnam VN30F futures):

1. **Replace `generate_synthetic_lob` with a tick parser**:
   * Ingest microsecond Level II quotes/transactions.
   * Aggregate into 1-second snapshots: recording cumulative market order volumes (`dmb`, `dms`) and top-of-book depths (`bid_q`, `ask_q`).
2. **Calibrate Friction Parameters**:
   * Incorporate exchange fees and latency delay between quote submission and order matching.
3. **Execution Verification**:
   * Run the Purged & Embargoed walk-forward evaluation protocol to monitor policy decay across macroeconomic regimes.
